# Drift correction - Cedric dataset_1, multiple tilts

Runs drift correction on a representative spread of tilt angles.
Shows all intermediate images for each tilt.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import quantem as em

In [ ]:
DATA_DIR = "/home/owner/data/cedric/drift/20260207_samsung_GAAFET/dataset_1"

images_0deg  = np.load(f"{DATA_DIR}/0_deg_images.npy")
images_90deg = np.load(f"{DATA_DIR}/90_deg_images.npy")
tilt_angles  = np.load(f"{DATA_DIR}/tilt_angles.npy")

# Representative spread across positive and negative tilts
TILT_INDICES = [0, 6, 12, 18, 23, 24, 30, 36, 42, 48]
angle_labels = [f"{int(tilt_angles[i]):+d} deg" for i in TILT_INDICES]

print(f"Selected tilts: {list(zip(TILT_INDICES, angle_labels))}")

In [ ]:
import numpy as np

tilt_images_a = images_0deg[TILT_INDICES]
tilt_images_b = images_90deg[TILT_INDICES]

corrected_stack, drift_objects = em.imaging.correct_series(
    tilt_images_a, tilt_images_b,
    scan_direction_degrees=[0, -90],
    preprocess=dict(pad_fraction=0.25, pad_value="median", kde_sigma=0.5, number_knots=1),
    align_affine=dict(step=0.02, num_tests=11),
    generate=dict(upsample_factor=1, kde_sigma=0.5),
)

for i, tilt_idx in enumerate(TILT_INDICES):
    angle = int(tilt_angles[tilt_idx])
    im0 = tilt_images_a[i]
    corrected = corrected_stack[i]
    diff = np.abs(im0.astype(float) - corrected)
    em.visualization.show_2d(
        [im0, corrected, diff],
        cmap=["gray", "gray", "inferno"],
        axsize=(6, 6),
        title=[
            f"[{angle:+d} deg] raw",
            f"[{angle:+d} deg] corrected",
            f"[{angle:+d} deg] |difference|  mean={diff.mean():.0f}",
        ],
    )

print("Done.")